# Retrieval-augmented generation (RAG)

A language model knows what was in its training data. It does not know your
company handbook, last week's tickets, or anything about a village that exists
only in your database. Ask it anyway and you get a confident guess.

**Retrieval-augmented generation** fixes that without retraining anything:

1. **Retrieve** the handful of passages most relevant to the question.
2. **Generate** an answer from those passages, handed to the model in its prompt.

The retrieval half runs on **embeddings** — vectors positioned so that text with
similar meaning lands close together (see {doc}`llm_clients`). Store those
vectors, and finding relevant passages becomes a nearest-neighbour search
instead of a keyword match.

Kaval.AI gives you a `RagService` that handles the whole loop:

- **Indexing** — embed text and persist it.
- **Querying** — embed a question and return the closest items, ranked by
  cosine similarity.
- **Organising** — group items into *collections*, tag them with a `source_id`
  and arbitrary JSON metadata, and filter retrieval on either.

Two backends implement it: {class}`~kavalai.PostgresRagService` (pgvector, for
production) and {class}`~kavalai.SqliteRagService` (a single portable file).
This notebook uses Postgres, then shows the SQLite one at the end.

## Setup

`KAVALAI_DB_URI` points at a PostgreSQL instance with the `pgvector` extension;
`KAVALAI_DB_SCHEMA` says which schema to use. The service creates and migrates
its own tables, so nothing needs provisioning by hand.

In [1]:
import os
import sys

import dotenv
from loguru import logger

dotenv.load_dotenv("../.env")

logger.remove()
_ = logger.add(sys.stderr, level="WARNING")

### Choosing an embedding model

An embedding model is named `provider/model`, like a chat model. `openai`,
`gemini`, `ollama` and `fastembed` are supported. This notebook uses
**`fastembed`**, which runs locally with no API key, so it is reproducible.
(The model downloads from the Hugging Face Hub on first use.)

:::{important}
Index and query with the **same** embedding model. Vectors from two different
models are not comparable, and the search silently returns nonsense rather than
failing.
:::

Each collection lives in its own table, provisioned on first index — the
embedding dimension is taken from the first batch.

In [2]:
from kavalai.rag import PostgresRagService

EMBEDDING_MODEL = "fastembed/BAAI/bge-small-en-v1.5"
COLLECTION = "green_village"

rag = PostgresRagService.from_uri(
    os.environ["KAVALAI_DB_URI"],
    model=EMBEDDING_MODEL,
    schema=os.environ.get("KAVALAI_DB_SCHEMA"),
)

## Indexing a corpus

Our corpus is the collected knowledge of Green Village — 104 residents, one pub,
one dalmatian.

`index_batch` embeds every text and stores it in one round trip. Each item
carries:

- **`texts`** — the content to embed and store;
- **`metadata_list`** — a JSON dict per item, for filtering and bookkeeping;
- **`source_ids`** — an external identifier per item (a document id, say). Chunks
  of one document share a `source_id`;
- **`collection_name`** — the logical group, which maps to a table.

For a single item there is `index(text, ...)`.

In [3]:
FACTS = [
    "President of Green Village is Thomas Cook (born 12.04.1994).",
    "Green Village has 104 residents.",
    "Green Village was founded on 03.09.1887 by shepherd Elias Thornbury.",
    "The tallest building in Green Village is the Old Grain Tower "
    "at 23 metres.",
    "Green Village's official flower is the marsh marigold.",
    "The village bakery, run by Greta Lindqvist, sells exactly 340 "
    "loaves every week.",
    "Green Village has one school with 14 pupils and 2 teachers.",
    "The annual Turnip Festival takes place every year on the third "
    "Saturday of October.",
    "Green Village's fire brigade consists of 7 volunteers and one "
    "dalmatian named Pepper.",
    "The village pond, Lake Miller, is 1.2 metres deep at its deepest point.",
    "Green Village's oldest resident is Agnes Whitlow (born 02.06.1929).",
    "The village has 3 streets: Main Road, Willow Lane, and Cobbler's Path.",
    "The local church bell weighs 412 kilograms and was cast in 1901.",
    "Green Village produces 8 tons of honey per year from its 26 beehives.",
    "The village library owns 1,847 books and is open on Tuesdays and Fridays.",
    "The speed limit everywhere in Green Village is 30 km/h.",
    "Green Village's only pub, The Rusty Anchor, has been operating "
    "since 1923.",
]

TOPICS = [
    "people", "people", "history", "buildings", "nature", "business", "school",
    "events", "safety", "nature", "people", "streets", "buildings", "business",
    "culture", "traffic", "business",
]

rows = await rag.index_batch(
    texts=FACTS,
    metadata_list=[{"topic": topic} for topic in TOPICS],
    source_ids=[f"fact-{i:02d}" for i in range(len(FACTS))],
    collection_name=COLLECTION,
)

print(f"indexed {len(rows)} facts into collection {COLLECTION!r}")

indexed 17 facts into collection 'green_village'


## Querying

`query` embeds the question and returns the `top_k` closest items, ordered from
most to least similar.

Retrieval is **semantic**, not lexical. The question below never says "oldest",
"born" or "Agnes", and a keyword search for "lived the longest" would match
nothing at all — yet the right fact comes back first.

In [4]:
hits = await rag.query(
    "Who has lived in the village the longest?",
    top_k=4,
    collection_name=COLLECTION,
)

for hit in hits:
    print(f"{hit.similarity:.3f}  [{hit.source_id}]  {hit.content}")

0.676  [fact-10]  Green Village's oldest resident is Agnes Whitlow (born 02.06.1929).
0.628  [fact-01]  Green Village has 104 residents.
0.576  [fact-11]  The village has 3 streets: Main Road, Willow Lane, and Cobbler's Path.
0.558  [fact-04]  Green Village's official flower is the marsh marigold.


Every hit is a {class}`~kavalai.RagServiceResult`, carrying the stored content
plus everything you need to filter, cite or debug it:

In [5]:
top = hits[0]
print("source_id   :", top.source_id)
print("similarity  :", round(top.similarity, 4))
print("collection  :", top.collection_name)
print("metadata    :", top.rag_metadata)
print("model       :", top.model)
print("dimensions  :", top.embedding_size)

source_id   : fact-10
similarity  : 0.6762
collection  : green_village
metadata    : {'topic': 'people'}
model       : fastembed/BAAI/bge-small-en-v1.5
dimensions  : 384


## The generation half

Retrieval on its own is useful — but the point is to answer questions. First,
watch what happens **without** retrieval. Green Village is fictional, so the
model has nothing to draw on:

In [6]:
from kavalai import make_client

client = make_client("openai/gpt-5.4-mini")

QUESTION = ("How old was Green Village's oldest resident at the "
            "2025 Turnip Festival?")

print(await client.prompt(QUESTION + " Answer in one sentence."))

I’m sorry, but I don’t have enough information to determine that.


Now retrieve first, and hand the model only what it needs. This is the entire
RAG pattern — a search, a prompt template, and a call:

In [7]:
PROMPT = """Answer the question using the facts below.

Every village-specific detail must come from these facts — do not invent any.
You may use general knowledge such as the calendar to reason about them. If the
facts are not enough to answer, say exactly what is missing.

Facts:
{context}

Question: {question}
"""


async def answer_with_rag(question: str, top_k: int) -> str:
    retrieved = await rag.query(
        question, top_k=top_k, collection_name=COLLECTION
    )
    print(f"retrieved (top_k={top_k}):")
    for hit in retrieved:
        print(f"  {hit.similarity:.3f}  {hit.content}")
    context = "\n".join(f"- {hit.content}" for hit in retrieved)
    return await client.prompt(
        PROMPT.format(context=context, question=question)
    )


print("\nanswer:\n" + await answer_with_rag(QUESTION, top_k=5))

retrieved (top_k=5):
  0.787  Green Village's oldest resident is Agnes Whitlow (born 02.06.1929).
  0.700  Green Village has 104 residents.
  0.681  President of Green Village is Thomas Cook (born 12.04.1994).
  0.668  Green Village was founded on 03.09.1887 by shepherd Elias Thornbury.
  0.615  Green Village's official flower is the marsh marigold.



answer:
Green Village’s oldest resident is Agnes Whitlow, born 02.06.1929.

At the 2025 Turnip Festival, she would have been **96 years old** if the festival was held on or after **02.06.2025**; otherwise, she would have been **95**.

What’s missing: the exact date of the 2025 Turnip Festival.


Not the answer we wanted — but exactly the behaviour we asked for. Answering
this needs *two* facts: Agnes Whitlow's birth date **and** the date of the
Turnip Festival. The top 5 hits carried only the first, so the model said so
instead of inventing a date.

It hedged and named exactly what was missing, rather than inventing a date.

This is the single most important property of RAG to understand: **the answer
can only be as good as the retrieval**. Widen the search and the second fact
comes along:

In [8]:
print("\nanswer:\n" + await answer_with_rag(QUESTION, top_k=8))

retrieved (top_k=8):
  0.787  Green Village's oldest resident is Agnes Whitlow (born 02.06.1929).
  0.700  Green Village has 104 residents.
  0.681  President of Green Village is Thomas Cook (born 12.04.1994).
  0.668  Green Village was founded on 03.09.1887 by shepherd Elias Thornbury.
  0.615  Green Village's official flower is the marsh marigold.
  0.615  The annual Turnip Festival takes place every year on the third Saturday of October.
  0.588  The tallest building in Green Village is the Old Grain Tower at 23 metres.
  0.574  Green Village has one school with 14 pupils and 2 teachers.



answer:
At the 2025 Turnip Festival, Green Village’s oldest resident, Agnes Whitlow, was **96 years old**.

Reasoning: she was born on **02.06.1929**, and the Turnip Festival is on the **third Saturday of October**. In **2025**, that date falls after her birthday, so she had already turned 96.


Now the model has both facts, and does the part it is good at — the third
Saturday of October 2025, minus a birthday in June 1929. Every *fact* came from
your database; the model supplied the language and the arithmetic.

That division of labour is the whole idea. It also explains the usual failure
mode: when a RAG system answers badly, the retrieval is usually at fault, not
the model. Tune `top_k`, the chunking, and the embedding model before you blame
the prompt.

Note also the "say so" instruction. Grounding a model reduces invention; it does
not abolish it.

## Collections, metadata and filters

A collection is, quite literally, a table: under PostgreSQL each collection gets its own
typed vector column and its own HNSW index, registered in a `rag_collections`
table, while the SQLite backend keeps every collection in one file with
`collection_name` as a column. Both layouts are described in
{doc}`../guides/data_model`. Ask the service what it is holding:

In [9]:
for collection in await rag.list_collections():
    print(collection)

print("\nentries in this collection:", await rag.count_entries(COLLECTION))

{'name': 'green_village', 'model': 'fastembed/BAAI/bge-small-en-v1.5', 'embedding_size': 384, 'schema_version': 1, 'count': 17}

entries in this collection: 17


`source_ids` narrows a search to specific documents — useful once you have
pre-filtered candidates some other way (by metadata, permissions, recency):

In [10]:
narrowed = await rag.query(
    "how tall is it?",
    top_k=3,
    collection_name=COLLECTION,
    source_ids=["fact-03", "fact-12"],  # grain tower, church bell
)
for hit in narrowed:
    print(f"{hit.similarity:.3f}  [{hit.source_id}]  {hit.content}")

0.485  [fact-03]  The tallest building in Green Village is the Old Grain Tower at 23 metres.
0.450  [fact-12]  The local church bell weighs 412 kilograms and was cast in 1901.


## Chunked documents and `keep_best`

Long documents get split into chunks that share one `source_id`. A plain query
can then return several chunks of the *same* document, crowding everything else
out of the top-k. `keep_best=True` collapses them to the best chunk per
`source_id`.

In [11]:
CHUNKS = [
    "The Rusty Anchor, chapter 1: the pub opened in 1923 in a "
    "former grain store.",
    "The Rusty Anchor, chapter 2: its oak bar was carved by Elias "
    "Thornbury's grandson.",
    "The Rusty Anchor, chapter 3: the pub serves 40 pints on a typical Friday.",
]

await rag.index_batch(
    texts=CHUNKS,
    metadata_list=[{"chunk": i} for i in range(len(CHUNKS))],
    source_ids=["pub-history"] * len(CHUNKS),
    collection_name=COLLECTION,
)

question = "Tell me about the village pub."
crowded = await rag.query(question, top_k=4, collection_name=COLLECTION)
collapsed = await rag.query(
    question, top_k=4, collection_name=COLLECTION, keep_best=True
)

print("without keep_best:", [h.source_id for h in crowded])
print("with keep_best   :", [h.source_id for h in collapsed])

without keep_best: ['pub-history', 'fact-16', 'pub-history', 'fact-01']
with keep_best   : ['pub-history', 'fact-16', 'fact-01', 'fact-11']


## Batched queries

Several questions at once? `query_batch` embeds and searches them in a single
database round trip (a `CROSS JOIN LATERAL` over the query vectors), returning
one result list per query, in input order.

In [12]:
questions = ["What do the bees produce?", "When is the library open?"]

for question, results in zip(
    questions,
    await rag.query_batch(
        questions, top_k=2, collection_name=COLLECTION
    ),
):
    print(f"{question!r}")
    for hit in results:
        print(f"   {hit.similarity:.3f}  {hit.content}")

'What do the bees produce?'
   0.618  Green Village produces 8 tons of honey per year from its 26 beehives.
   0.479  Green Village's official flower is the marsh marigold.
'When is the library open?'
   0.624  The village library owns 1,847 books and is open on Tuesdays and Fridays.
   0.479  The Rusty Anchor, chapter 1: the pub opened in 1923 in a former grain store.


## Similarity matrix

`compute_similarity_matrix` scores every text against every listed `source_id`
and returns a 2-D matrix — useful for ranking, clustering or deduplication, where
all pairwise scores are needed at once rather than a top-k.

In [13]:
texts = ["honey production", "opening hours", "village leadership"]
targets = ["fact-13", "fact-14", "fact-00"]  # bees, library, president

matrix = await rag.compute_similarity_matrix(
    texts=texts, source_ids=targets, collection_name=COLLECTION
)

print(f"{'':>20}" + "".join(f"{t:>10}" for t in targets))
for text, row in zip(texts, matrix):
    print(f"{text:>20}" + "".join(f"{value:10.3f}" for value in row))

                       fact-13   fact-14   fact-00
    honey production     0.657     0.337     0.415
       opening hours     0.420     0.568     0.467
  village leadership     0.548     0.558     0.663


The diagonal dominates — each phrase scores highest against the fact it is
about.

## A portable index: SQLite

{class}`~kavalai.SqliteRagService` implements the same interface over a single
SQLite file (using the `sqlite-vector` extension). No server, and the file is
readable in the browser via SQLite AI's WASM build — so you can build an index
offline and ship it with a static site. See {doc}`run_in_browser`.

The API is identical; only the constructor changes.

In [14]:
from kavalai.rag import SqliteRagService

portable = SqliteRagService(":memory:", model=EMBEDDING_MODEL)

await portable.index_batch(
    texts=FACTS[:6],
    metadata_list=[{"topic": topic} for topic in TOPICS[:6]],
    source_ids=[f"fact-{i:02d}" for i in range(6)],
)

for hit in await portable.query("who bakes the bread?", top_k=2):
    print(f"{hit.similarity:.3f}  {hit.content}")

0.530  The village bakery, run by Greta Lindqvist, sells exactly 340 loaves every week.
0.506  President of Green Village is Thomas Cook (born 12.04.1994).


Pass a path instead of `":memory:"` and you get a file you can copy, version or
serve.

## Retrieval inside a workflow

Everything so far has been Python calling a `RagService` directly. A workflow
can do the retrieval itself, with a **`rag_query` node** — so the search becomes
part of the document, shows up in the diagram, and is recorded as its own task
alongside the model call.

The node is **read-only**: it calls `query` and nothing else. Indexing stays in
Python, where you can see what you are writing and to which collection.

`query` is a template, exactly like an `llm` node's `prompt`, so it can pull the
question straight out of the run context. `store="content"` keeps just the hit
texts — which is what the next prompt wants; the default, `store="results"`,
keeps the full {class}`~kavalai.RagServiceResult` list so scores and metadata
stay available for routing.

The service is passed to the engine, not named in the document: a workflow is
served over HTTP and edited in the backoffice, so it carries a *name*, never a
connection string. With one index you do not even need the name — a single
service is registered as `"default"`, which is what a node resolves to when
neither it nor the workflow says otherwise.

In [ ]:
from kavalai import WorkflowBuilder

village_index = SqliteRagService(":memory:", model=EMBEDDING_MODEL)
await village_index.index_batch(
    texts=FACTS,
    metadata_list=[{"topic": topic} for topic in TOPICS],
    source_ids=[f"fact-{i:02d}" for i in range(len(FACTS))],
)

ANSWER_PROMPT = """Answer the question using the facts below.

Every village-specific detail must come from these facts — do not invent any.
You may use general knowledge such as the calendar to reason about them.

Facts:
{{ context.facts }}

Question: {{ context.input.question }}
"""

village_desk = (
    WorkflowBuilder("green-village-desk", llm_model="openai/gpt-5.4-mini")
    .data_type("input", {"question": str})
    .data_type("output", {"answer": str})
    .start(next="retrieve")
    .rag_query(
        "retrieve",
        query="{{ context.input.question }}",
        output="facts",
        next="answer",
        top_k=8,
        store="content",
    )
    .llm("answer", prompt=ANSWER_PROMPT, output="output", next="done")
    .end(name="done", output="output")
    .build_engine(rag_services=village_index)
)

state = await village_desk.run({"question": QUESTION})

print("trace:  ", " -> ".join(state.trace))
print("answer: ", state.output_data["answer"])

trace:   start -> retrieve -> answer -> done
answer:  Agnes Whitlow, born 02.06.1929, was 96 years old at the 2025 Turnip Festival.


Two nodes, and the whole RAG loop is a document. The retrieval step is visible
in the rendered graph, and the run recorded a task for it — how long the search
took and what it returned — next to the model call it fed.

To point several nodes at different indexes, pass a dict
(`rag_services={"handbook": ..., "tickets": ...}`) and name one per node with
`service=`, or set a workflow-wide default with
`WorkflowBuilder(..., rag_service="handbook")`. Under
`python -m kavalai.server`, where nobody constructs the engine, register the
service instead — see {doc}`../cookbook/index`.

## Cleanup

`drop_collection` removes the collection's table and its registry entry.
`delete_by_source_id` removes individual documents.

In [15]:
await rag.drop_collection(COLLECTION)
print("collections now:", [c["name"] for c in await rag.list_collections()])

collections now: []


## Where to next

- {doc}`llm_clients` — the embedding clients underneath, and similarity in the raw.
- {doc}`workflow` — the `rag_query` node above, alongside every other node type.
- {doc}`agents` — let an agent decide *when* to search.
- {doc}`run_in_browser` — ship a pre-built index and query it client-side.
- {doc}`../ui/index` — browse and query collections in the backoffice RAG explorer.
- {doc}`../guides/data_model` — the tables a retrieval index is stored in.